# Papermill driver for 001_Preprocessing per sample

Runs 001_Preprocessing once for each included sample and collects the results.

**Input**
- Metadata/samples.csv (created by 000_Raw_Data_And_Cohort)
- Raw counts in the folder set by CAP_RAW_DATA

**Output**
- runs/001/<smaple_id>.ipynb: executed notebook for each sample
- data/Adata_Saves/Interim_head/<sample_id>.h5ad: written by 001
- results/dataframa_saves/qc/<sample_id>.csv: written by 001, collected below into results/qc_summary.csv

**Options**
- --demo: runs only the two demo samples
- --force: reruns samples that already have a finished notebook

**Failure handling**
A failed sample keeps its .partial.ipynb and its log. If any sample fails, the driver stops with an error and the QC summary is not written.

## Setup

In [1]:
import argparse
import pandas as pd
import papermill as pm

In [2]:
from cap.paths import (project_root, display_path, raw_data_dir, notebook_dir, sample_sheet, dataframe_dir, runs_dir, log_dir)
pd.set_option("display.width", 140)
pd.set_option("display.max_rows", 80)
# Resolve all paths through cap.paths
root = project_root()
raw_dir = raw_data_dir()
notebooks_dir = notebook_dir()
out_dir = runs_dir("001_Preprocessing")
log_dir = log_dir()
print("Notebook directory:", display_path(notebooks_dir))

Notebook directory: Notebooks


In [3]:
DEMO_IDS = [
    "GSM4909292_HER2-MH0069",
    "GSM4909272_N-MH0021-Total"]
KERNEL = "cap"

parser = argparse.ArgumentParser()
parser.add_argument("--demo", action="store_true", help="run only DEMO_IDS")
parser.add_argument("--force", action="store_true", help="rerun finished samples")
args, _ = parser.parse_known_args()
#args, _ = parser.parse_known_args(["--demo"])#uncomment this to run demo version

notebook = notebooks_dir / "001_Preprocessing.ipynb"

In [4]:
from cap.metadata import load_sample_sheet
sheet = load_sample_sheet()                 # validated, indexed by sample_id
ids = sheet.index[sheet["include"]].tolist()

In [5]:
done_ids, skipped, failed = [], [], []
for sid in ids:
    done = out_dir / f"{sid}.ipynb"
    partial = out_dir / f"{sid}.partial.ipynb"
    log = log_dir / f"{sid}.log"

    if done.exists() and not args.force:
        skipped.append(sid)
        print(f"skip {sid}")
        continue

    print(f"run  {sid}")
    try:
        # Papermill writes all cell output and errors of this run to the per-sample log
        with open(log, "w") as fh:
            pm.execute_notebook(notebook, partial,
                                parameters={"sample_id": sid},
                                kernel_name=KERNEL, progress_bar=False,
                                log_output=True, stdout_file=fh, stderr_file=fh)
        partial.replace(done)  # only a successful run counts as done
        done_ids.append(sid)
    except Exception as exc:
        failed.append(sid)
        print(f"FAILED {sid}: {type(exc).__name__}: {exc} (log: {display_path(log)})")

print(f"\nrun: {len(done_ids)}  skipped: {len(skipped)}  failed: {len(failed)}  "
      f"(of {len(ids)} selected)")
if failed:
    print("failed:", ", ".join(failed))
    raise RuntimeError(f"{len(failed)} sample(s) failed: {', '.join(failed)}")

skip GSM4909295_ER-AH0319
skip GSM4909296_ER-MH0001
skip GSM4909302_ER-MH0025
skip GSM4909305_ER-MH0029-7C
skip GSM4909306_ER-MH0029-9C
skip GSM4909300_ER-MH0032
skip GSM4909307_ER-MH0040
skip GSM4909301_ER-MH0042
skip GSM4909309_ER-MH0043-T
skip GSM4909311_ER-MH0056-T
skip GSM4909313_ER-MH0064-T
run  GSM4909299_ER-MH0114-T3


/Users/gernotstuhler/miniforge3_arm64/envs/conda_py312/lib/python3.12/site-packages/anndata/_core/anndata.py:1880: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")

/Users/gernotstuhler/miniforge3_arm64/envs/conda_py312/lib/python3.12/site-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  mod_version = _find_version(mod.__version__)
/Users/gernotstuhler/miniforge3_arm64/envs/conda_py312/lib/python3.12/site-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  mod_version = _find_version(mod.__version__)



run  GSM4909297_ER-MH0125


/Users/gernotstuhler/miniforge3_arm64/envs/conda_py312/lib/python3.12/site-packages/anndata/_core/anndata.py:1880: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")

/Users/gernotstuhler/miniforge3_arm64/envs/conda_py312/lib/python3.12/site-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  mod_version = _find_version(mod.__version__)
/Users/gernotstuhler/miniforge3_arm64/envs/conda_py312/lib/python3.12/site-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  mod_version = _find_version(mod.__version__)



run  GSM4909303_ER-MH0151


...

run  GSM4909283_TN-SH0106


/Users/gernotstuhler/miniforge3_arm64/envs/conda_py312/lib/python3.12/site-packages/anndata/_core/anndata.py:1880: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")

/Users/gernotstuhler/miniforge3_arm64/envs/conda_py312/lib/python3.12/site-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  mod_version = _find_version(mod.__version__)
/Users/gernotstuhler/miniforge3_arm64/envs/conda_py312/lib/python3.12/site-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  mod_version = _find_version(mod.__version__)



run  GSM4909287_TN-B1-Tum0554


/Users/gernotstuhler/miniforge3_arm64/envs/conda_py312/lib/python3.12/site-packages/anndata/_core/anndata.py:1880: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")

/Users/gernotstuhler/miniforge3_arm64/envs/conda_py312/lib/python3.12/site-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  mod_version = _find_version(mod.__version__)
/Users/gernotstuhler/miniforge3_arm64/envs/conda_py312/lib/python3.12/site-packages/session_info/main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  mod_version = _find_version(mod.__version__)




run: 34  skipped: 11  failed: 0  (of 45 selected)


In [6]:
# Supplementary QC table: one row per included sample.
# Runs only if the loop above finished without failures (see RuntimeError).
results_dir = project_root() / "results"
results_dir.mkdir(parents=True, exist_ok=True)
qc_dir = dataframe_dir()

missing = [sid for sid in ids if not (qc_dir / f"qc_{sid}.csv").is_file()]
if missing:
    raise FileNotFoundError(f"QC tables missing for {len(missing)} sample(s): {missing}")

qc_summary = (
    pd.concat([pd.read_csv(qc_dir / f"qc_{sid}.csv") for sid in ids], ignore_index=True)
    .sort_values(["condition", "sample"])
    .reset_index(drop=True)
)

# A demo run must not overwrite the full supplementary table
suffix = "_demo" if args.demo else ""
out_file = results_dir / f"qc_summary{suffix}.csv"
qc_summary.to_csv(out_file, index=False)

if not args.demo:
    expected_n = int(sheet["include"].sum())  # taken from metadata, not hard-coded
    assert len(qc_summary) == expected_n, (len(qc_summary), expected_n)
    assert qc_summary["sample"].is_unique, "duplicate sample rows in QC summary"

print(f"written: {display_path(out_file)} ({len(qc_summary)} samples)")

written: results/qc_summary.csv (45 samples)
